# Pre-earnings hedge timing — `next_earnings` + `decompose` (H.89.6)

**[Open in Colab](https://colab.research.google.com/github/BlueWaterCorp/RiskModels_API/blob/main/sdk/notebooks/pre_earnings_hedge_timing.ipynb)** · **[Get API key](https://riskmodels.app/get-key)**

Idiosyncratic risk spikes around earnings. If you want exposure to the earnings *event itself* (the residual/stock-specific bet) without the market/sector beta riding along for the same few days, you need two things: **when** the event is likely, and **how much** market/sector exposure to lay off going into it.

- `client.next_earnings(ticker)` — a **cadence-based estimate**, not a confirmed earnings-calendar date. It projects from the median gap between this ticker's own historical `filed_date` values. There is no analyst-estimate or vendor earnings-calendar feed in the store (`eps_forecast` / `analyst_rating` / `target_price` are permanently held back — see `lib/api/fundamentals-contract.ts` `FUNDAMENTALS_HELD_BACK_FIELDS` in the API repo, and `CLAUDE.md`'s derived-only licensing rule). Treat the date as a planning window, not a forecast.
- `client.decompose(ticker, as_dataframe=True)` — current per-layer hedge ratios and recommended hedge ETFs, to structure the market/sector leg of the hedge.


### Colab only (skip locally)

Run once to install the SDK; local users should `pip install riskmodels-py` in their venv.


In [1]:
import sys

try:
    import google.colab  # noqa: F401
    _COLAB = True
except ImportError:
    _COLAB = False

if _COLAB:
    import subprocess

    _deps = ["python-dotenv"]
    _pypi = "riskmodels-py>=0.3.4"
    _git = (
        "riskmodels-py @ git+https://github.com/BlueWaterCorp/RiskModels_API.git"
        "@main#subdirectory=sdk"
    )
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", _pypi, *_deps],
            stdout=subprocess.DEVNULL,
        )
        print("Colab: installed riskmodels-py from PyPI (+ python-dotenv).")
    except subprocess.CalledProcessError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", _git, *_deps],
            stdout=subprocess.DEVNULL,
        )
        print(
            "Colab: installed riskmodels-py from GitHub main "
            "(PyPI may not list this version yet)."
        )
else:
    print("Local: use your environment (pip install riskmodels-py python-dotenv).")


Local: use your environment (pip install riskmodels-py python-dotenv).


## 1. Connect — `RiskModelsClient.from_env()`

Loads **`RISKMODELS_API_KEY`** from shell env, `.env` / `.env.local` (when `python-dotenv` is installed), or Colab Secrets.


In [2]:
import os

from IPython.display import display

from riskmodels import RiskModelsClient
from riskmodels.client import DEFAULT_BASE_URL
from riskmodels.notebook import load_notebook_dotenv

load_notebook_dotenv()
print("RISKMODELS_BASE_URL =", os.environ.get("RISKMODELS_BASE_URL", DEFAULT_BASE_URL))
client = RiskModelsClient.from_env()


RISKMODELS_BASE_URL = https://riskmodels.app/api


## 2. When's the next print?


In [3]:
TICKER = "NVDA"

next_earn = client.next_earnings(TICKER)
next_earn


{'ticker': 'NVDA',
 'last_filed_date': '2026-05-27',
 'estimated_next_filed_date': '2026-08-26',
 'median_cadence_days': 91,
 'n_periods_observed': 8,
 'basis': 'filed_date_cadence',
 'note': 'Statistical projection from historical filing cadence, not a confirmed earnings-calendar date. No analyst estimates or vendor earnings-calendar feed are stored (derived-only licensing posture).'}

## 3. Days until the estimated window

A simple planning threshold: inside ~21 trading days of the estimate, start thinking about hedging the beta legs; the exact cutoff is a judgment call, not something the API prescribes.


In [4]:
from datetime import date

next_date_str = next_earn.get("estimated_next_filed_date")
if next_date_str is None:
    print(
        f"Not enough filing history to project a cadence for {TICKER} "
        "(need >= 2 filed quarters)."
    )
    days_until = None
else:
    days_until = (date.fromisoformat(next_date_str) - date.today()).days
    print(f"{TICKER}: ~{days_until} calendar days to the estimated next filing ({next_date_str}).")
    print(f"Basis: {next_earn['basis']} over {next_earn['n_periods_observed']} observed quarters.")


NVDA: ~50 calendar days to the estimated next filing (2026-08-26).
Basis: filed_date_cadence over 8 observed quarters.


## 4. Structure the beta hedge

`decompose` gives the market/sector/subsector hedge ratios and which ETF trades each one. The `residual` layer is deliberately **not hedgeable** here — that's the earnings-event exposure you're choosing to keep.


In [5]:
hedge_df = client.decompose(TICKER, as_dataframe=True)
display(hedge_df)

hedgeable = hedge_df[hedge_df["hedgeable"]]
print("Layers to hedge going into the print (market/sector/subsector):")
display(hedgeable[["layer", "hr", "hedge_etf"]])
print("\nKept as the earnings bet (not hedged):")
display(hedge_df[hedge_df["layer"] == "residual"][["layer", "er"]])


,ticker,layer,er,hr,hedge_etf,hedgeable,data_as_of
0,NVDA,market,0.419527,-0.961814,SPY,True,2026-07-06
1,NVDA,sector,0.130094,-0.086165,XLK,True,2026-07-06
2,NVDA,subsector,-0.012480,-0.359021,SMH,True,2026-07-06
3,NVDA,residual,0.462859,NaN,None,False,2026-07-06
4,NVDA,style,0.132342,NaN,None,False,2026-07-06
5,NVDA,stock_specific,0.449242,NaN,None,False,2026-07-06


Layers to hedge going into the print (market/sector/subsector):


,layer,hr,hedge_etf
0,market,-0.961814,SPY
1,sector,-0.086165,XLK
2,subsector,-0.359021,SMH



Kept as the earnings bet (not hedged):


,layer,er
3,residual,0.462859


## 5. Put it together


In [6]:
if days_until is not None:
    if days_until <= 21:
        print(f"Inside the pre-earnings window for {TICKER} — consider laying off the")
        print("hedgeable layers above via their listed hedge_etf before the print, sized")
        print("to each layer's hr, to isolate the residual/stock-specific earnings bet.")
    else:
        print(f"{TICKER} is not yet inside a ~21-day pre-earnings window; re-check closer to")
        print(f"{next_date_str}.")


NVDA is not yet inside a ~21-day pre-earnings window; re-check closer to
2026-08-26.


## Caveats

- `next_earnings` is a **statistical projection from cadence**, not a confirmed date — companies change reporting schedules, and the median-gap estimate has no confidence interval attached. Always cross-check against a real calendar before sizing a trade on it.
- Hedge ratios from `decompose` are point-in-time (`data_as_of`) and drift as the regression windows roll forward — re-fetch close to execution, not once days in advance.
- Sign convention: `hedge[etf] == -exposure[layer].hr` — a positive stock `hr` means **short** the ETF to hedge a long position.

## Next steps

- **Quality-overlay screen:** [`quality_overlay_screen.ipynb`](./quality_overlay_screen.ipynb)
- **PIT fundamentals event study:** [`pit_fundamentals_event_study.ipynb`](./pit_fundamentals_event_study.ipynb)
